# Gemini: Aspect Identification & Extraction Pipeline
This notebook runs general and specific aspect identification and aspect extraction pipelines, then generates validation results and visualizations.

Sections added below:
- Aspect Identification
  - General Aspects
  - Specific Aspects
- Aspect Extraction
- Visualization Generation

In [ ]:
import base64
import os
from pathlib import Path
from dotenv import load_dotenv
from gemini import generate, TaskType
import time

# Load environment variables
load_dotenv()

## Aspect Identification
This section contains code to identify general and specific aspects. The following cells set paths and run the general identification step and validation.

In [ ]:
# Configure file paths
IDENTIFIER_FILES = Path('d:/DLSU/THESIS/Aspect-Extraction-On-Shopee-Reviews/geminiAPI/aspect_identification')

# Files for General Aspect Identification
GEN_IDENTIFICATION_FILES = {
    'gen-example': IDENTIFIER_FILES / 'multi-gen-examples.csv',
    'gen-test': IDENTIFIER_FILES / 'review-only-test.csv',
    'gen-output': IDENTIFIER_FILES / 'final_predicted.csv'
}

### General Aspects — run and validate
The next code cell runs the general aspect identification model (LLM) using provided example and test files. The following validation cell runs `validate_all_aspects` to compute exact match accuracy, hamming loss, and F1 metrics for the general aspects.

In [ ]:
# Run Aspect Identification
print("=== Starting General Aspect Identification ===")
generate(
    TaskType.IDENTIFY_GENERAL,
    example_file=GEN_IDENTIFICATION_FILES['gen-example'],
    test_file=GEN_IDENTIFICATION_FILES['gen-test'],
    output_file=GEN_IDENTIFICATION_FILES['gen-output']
)
print("\n=== General Aspect Identification Complete ===")

#### Description: Running general identification and validation
- `generate(TaskType.IDENTIFY_GENERAL, ...)` runs the general aspect identification LLM pipeline using the example and test files.
- `validate_all_aspects(...)` computes the metrics and writes validation and error-analysis outputs.

These steps produce `final_predicted.csv`, `general_validation_results.txt`, and `general_error_analysis.txt` (files under the `aspect_identification` folder).

In [ ]:
# Validate results
ASPECTS = ['product', 'delivery', 'price', 'service']

print("\n=== Starting Validation ===")
from aspect_identification.validate_multiGen import validate_all_aspects
pred_file = GEN_IDENTIFICATION_FILES['gen-output']
gt_file = IDENTIFIER_FILES / "valid_multiGen.csv"
valid_results_file = IDENTIFIER_FILES / 'general_validation_results.txt'
error_analysis_file = IDENTIFIER_FILES / 'general_error_analysis.txt'

results = validate_all_aspects(pred_file, gt_file, ASPECTS,
                         valid_results_file, error_analysis_file)

print("\n=== Validation Complete ===")

### Specific Aspects — identification & validation
The following cells run specific-aspect identification for each general aspect (product, delivery, price, service). Each block configures example files, test files, and output paths for specific-aspect models. After each prediction, validation is run and error analysis files are written (see `validation-results` and `error-analysis` folders).

In [ ]:
SPECIFIC_IDENTIFY_FILES = Path('d:/DLSU/THESIS/Aspect-Extraction-On-Shopee-Reviews/geminiAPI/annotations/BTK_annotations/specific_aspect_identification')

SP_EXAMPLE_FILES = {
    'proSP-example': SPECIFIC_IDENTIFY_FILES / 'proSP-examples.csv',  # for product aspect extraction
    'delSP-example': SPECIFIC_IDENTIFY_FILES / 'delSP-examples.csv',   # for delivery aspect extraction
    'priSP-example': SPECIFIC_IDENTIFY_FILES / 'priSP-examples.csv',   # for price aspect extraction
    'serSP-example': SPECIFIC_IDENTIFY_FILES / 'serSP-examples.csv'   # for service aspect extraction
}

In [ ]:
sp_identify_config = {
        'product': {
            'task_type': TaskType.IDENTIFY_PRODUCT_SPECIFIC,
            'example_file': SP_EXAMPLE_FILES['proSP-example'],
            'test_file': IDENTIFIER_FILES / 'specific-aspects' / 'sp-pro-test.csv',
            'output_file': IDENTIFIER_FILES / 'specific-aspects' / 'sp-pro-pred.csv'
        },
        'delivery': {
            'task_type': TaskType.IDENTIFY_DELIVERY_SPECIFIC,
            'example_file': SP_EXAMPLE_FILES['delSP-example'],
            'test_file': IDENTIFIER_FILES / 'sp-del-test.csv',
            'output_file': IDENTIFIER_FILES / 'sp-del-pred.csv'
        },
        'price': {
            'task_type': TaskType.IDENTIFY_PRICE_SPECIFIC,
            'example_file': SP_EXAMPLE_FILES['priSP-example'],
            'test_file': IDENTIFIER_FILES / 'sp-pri-test.csv',
            'output_file': IDENTIFIER_FILES / 'sp-pri-pred.csv'
        },
        'service': {
            'task_type': TaskType.IDENTIFY_SERVICE_SPECIFIC,
            'example_file': SP_EXAMPLE_FILES['serSP-example'],
            'test_file': IDENTIFIER_FILES / 'sp-ser-test.csv',
            'output_file': IDENTIFIER_FILES / 'sp-ser-pred.csv'
        }
    }
# Run Specific Aspect Identification for one file/batch
# print("=== Starting Delivery Aspect Identification ===")
# generate(
#     TaskType.IDENTIFY_DELIVERY_SPECIFIC,
#     example_file=SP_EXAMPLE_FILES['delSP-example'],
#     test_file=IDENTIFIER_FILES / 'specific-aspects' / 'sp-del-test.csv',
#     output_file=IDENTIFIER_FILES / 'specific-aspects' / 'sp-del-pred.csv'
# )
# print("\n=== Delivery Aspect Identification Complete ===")

In [ ]:
# Run Aspect Identification for multiple files/batches of product specific aspect
SPECIFIC_PRO_DIR = Path('d:/DLSU/THESIS/Aspect-Extraction-On-Shopee-Reviews/geminiAPI/annotations/BTK_annotations/specific_aspect_identification/product_identify')

# for i in range(1, 5):
#     print(f"=== Starting Product Specific Aspect Identification Batch {i} ===")

#     test_file = SPECIFIC_PRO_DIR / f"proSP-retry{i}.csv"
#     output_file = SPECIFIC_PRO_DIR / f"proSP-2ndtry{i}.csv"

#     generate(
#         TaskType.IDENTIFY_PRODUCT_SPECIFIC,
#         example_file=SP_EXAMPLE_FILES['proSP-example'],
#         test_file=test_file,
#         output_file=output_file
#     )
#     print(f"✓ Batch {i} complete - saved to {output_file}")

#     if i < 4:
#         print(f"⏳ Waiting 6 seconds before next batch...")
#         time.sleep(6)

print("\n=== Product Specific Aspect Identification Complete ===")

In [ ]:
# Run Aspect Identification for multiple files/batches of delivery specific aspect
SPECIFIC_DEL_DIR = Path('d:/DLSU/THESIS/Aspect-Extraction-On-Shopee-Reviews/geminiAPI/annotations/BTK_annotations/specific_aspect_identification/delivery_identify')

for i in range(1, 2):
    print(f"=== Starting Delivery Specific Aspect Identification Batch {i} ===")

    test_file = SPECIFIC_DEL_DIR / f"delSP-retry{i}.csv"
    output_file = SPECIFIC_DEL_DIR / f"delSP-2ndtry{i}.csv"
    
    generate(
        TaskType.IDENTIFY_DELIVERY_SPECIFIC,
        example_file=SP_EXAMPLE_FILES['delSP-example'],
        test_file=test_file,
        output_file=output_file
    )
    print(f"✓ Batch {i} complete - saved to {output_file}")

    # if i < 5:
    #     print(f"⏳ Waiting 6 seconds before next batch...")
    #     time.sleep(6)

print("\n=== Delivery Specific Aspect Identification Complete ===")

In [ ]:
# Run Aspect Identification for multiple files/batches of price specific aspect
SPECIFIC_PRI_DIR = Path('d:/DLSU/THESIS/Aspect-Extraction-On-Shopee-Reviews/geminiAPI/annotations/BTK_annotations/specific_aspect_identification/price_identify')

for i in range(1, 9):
    print(f"=== Starting Price Specific Aspect Identification Batch {i} ===")

    test_file = SPECIFIC_PRI_DIR / f"{i}.csv"
    output_file = SPECIFIC_PRI_DIR / f"priSP-output{i}.csv"

    generate(
        TaskType.IDENTIFY_PRICE_SPECIFIC,
        example_file=SP_EXAMPLE_FILES['priSP-example'],
        test_file=test_file,
        output_file=output_file
    )
    print(f"✓ Batch {i} complete - saved to {output_file}")

    if i < 8:
        print(f"⏳ Waiting 6 seconds before next batch...")
        time.sleep(6)

print("\n=== Price Specific Aspect Identification Complete ===")

In [ ]:
# Run Aspect Identification for multiple files/batches of service specific aspect
SPECIFIC_SER_DIR = Path('d:/DLSU/THESIS/Aspect-Extraction-On-Shopee-Reviews/geminiAPI/annotations/BTK_annotations/specific_aspect_identification/service_identify')

for i in range(1, 2):
    print(f"=== Starting Service Specific Aspect Identification Batch {i} ===")

    test_file = SPECIFIC_SER_DIR / f"serSP-5thtry{i}.csv"
    output_file = SPECIFIC_SER_DIR / f"5thtry{i}.csv"

    generate(
        TaskType.IDENTIFY_SERVICE_SPECIFIC,
        example_file=SP_EXAMPLE_FILES['serSP-example'],
        test_file=test_file,
        output_file=output_file
    )
    print(f"✓ Batch {i} complete - saved to {output_file}")

    # if i < 2:
    #     print(f"⏳ Waiting 6 seconds before next batch...")
    #     time.sleep(6)
print("\n=== Service Specific Aspect Identification Complete ===")

In [ ]:
# Validate results
# specific: product
PRO_ASPECTS = ['color', 'condition', 'correctness', 'durability', 'effectiveness', 'functionality', 'material', 'sensory', 'measurement', 'general']
# specific: delivery
DEL_ASPECTS = ['condition', 'correctness', 'timeliness', 'general']
# specific: price
PRI_ASPECTS = ['affordability', 'value_for_money', 'general']
# specific: service
SER_ASPECTS = ['handling', 'responsiveness', 'trustworthiness', 'general']

validation_config = {
        'product': {
            'aspects': PRO_ASPECTS,
            'pred_file': IDENTIFIER_FILES / 'specific-aspects' / 'sp-pro-pred.csv',
            'gt_file': IDENTIFIER_FILES / 'specific-aspects' / 'sp-pro-gt.csv',
            'valid_results_file': IDENTIFIER_FILES / 'validation-results' / 'product_validation_results.txt',
            'error_results_file': IDENTIFIER_FILES / 'error-analysis' / 'product_error_analysis.txt'
        },
        'delivery': {
            'aspects': DEL_ASPECTS,
            'pred_file': IDENTIFIER_FILES / 'specific-aspects' / 'sp-del-pred.csv',
            'gt_file': IDENTIFIER_FILES / 'specific-aspects' / 'sp-del-gt.csv',
            'valid_results_file': IDENTIFIER_FILES / 'validation-results' / 'delivery_validation_results.txt',
            'error_results_file': IDENTIFIER_FILES / 'error-analysis' / 'delivery_error_analysis.txt'
        },
        'price': {
            'aspects': PRI_ASPECTS,
            'pred_file': IDENTIFIER_FILES / 'specific-aspects' / 'sp-pri-pred.csv',
            'gt_file': IDENTIFIER_FILES / 'specific-aspects' / 'sp-pri-gt.csv',
            'valid_results_file': IDENTIFIER_FILES / 'validation-results' / 'price_validation_results.txt',
            'error_results_file': IDENTIFIER_FILES / 'error-analysis' / 'price_error_analysis.txt'
        },
        'service': {
            'aspects': SER_ASPECTS,
            'pred_file': IDENTIFIER_FILES / 'specific-aspects' / 'sp-ser-pred.csv',
            'gt_file': IDENTIFIER_FILES / 'specific-aspects' / 'sp-ser-gt.csv',
            'valid_results_file': IDENTIFIER_FILES / 'validation-results' / 'service_validation_results.txt',
            'error_results_file': IDENTIFIER_FILES / 'error-analysis' / 'service_error_analysis.txt'
        }
    }

print("\n=== Starting Validation ===")
from aspect_identification.validate_multiGen import validate_all_aspects
for aspect, config in validation_config.items():
    ASPECTS = config['aspects']
    pred_file = config['pred_file']
    gt_file = config['gt_file']
    valid_results_file = config['valid_results_file']
    error_analysis_file = config['error_results_file']

    results = validate_all_aspects(pred_file, gt_file, ASPECTS,
                            valid_results_file, error_analysis_file)

print("\n=== Validation Complete ===")

In [ ]:
from aspect_identification.validate_multiGen import calculate_overall_performance, save_overall_results

general_aspect_mapping = {
    'product': ['color', 'condition', 'correctness', 'durability', 'effectiveness', 'functionality', 'material', 'sensory', 'measurement', 'general'],
    'delivery': ['condition', 'correctness', 'timeliness', 'general'],
    'price': ['affordability', 'value_for_money', 'general'],
    'service': ['handling', 'responsiveness', 'trustworthiness', 'general']
}

error_analysis_files = {
    'product': IDENTIFIER_FILES / 'error-analysis' / 'product_error_analysis.txt',
    'delivery': IDENTIFIER_FILES / 'error-analysis' / 'delivery_error_analysis.txt',
    'price': IDENTIFIER_FILES / 'error-analysis' / 'price_error_analysis.txt',
    'service': IDENTIFIER_FILES / 'error-analysis' / 'service_error_analysis.txt',
}

results = calculate_overall_performance(general_aspect_mapping, error_analysis_files)
save_overall_results(results, IDENTIFIER_FILES / 'overall_performance.txt')

## Aspect Extraction
This section runs aspect extraction models (product, delivery, price, service). Use the example and test files defined above to extract phrases/mentions from reviews. Extraction outputs are saved under the `aspect_extraction` folder.

In [ ]:
# Files for General Aspect Extraction
EXTRACTOR_FILES = Path('d:/DLSU/THESIS/Aspect-Extraction-On-Shopee-Reviews/geminiAPI/aspect_extraction')

EXTRACTION_FILES = {
    'prod-example': EXTRACTOR_FILES / 'product-examples.json',  # for product aspect extraction
    'prod-test': EXTRACTOR_FILES / 'product-test.csv',
    'prod-output': EXTRACTOR_FILES / 'product-predicted.json',
    'del-example': EXTRACTOR_FILES / 'delivery-examples.json',   # for delivery aspect extraction
    'del-test': EXTRACTOR_FILES / 'delivery-test.csv',
    'del-output': EXTRACTOR_FILES / 'delivery-predicted.csv',
    'pri-example': EXTRACTOR_FILES / 'price-examples.json',   # for price aspect extraction
    'pri-test': EXTRACTOR_FILES / 'price-test.csv',
    'pri-output': EXTRACTOR_FILES / 'price-predicted.json',
    'ser-example': EXTRACTOR_FILES / 'service-examples.json',   # for service aspect extraction
    'ser-test': EXTRACTOR_FILES / 'service-test.csv',
    'ser-output': EXTRACTOR_FILES / 'service-predicted.json'
}

In [ ]:
# Run Product Aspect Extraction (when needed)
def run_extraction():
    extraction_config = {
        'product': {
            'task_type': TaskType.EXTRACT_PRODUCT,
            'example_file': EXTRACTION_FILES['prod-example'],
            'test_file': EXTRACTION_FILES['prod-test'],
            'output_file': EXTRACTION_FILES['prod-output']
        }
        # 'delivery': {
        #     'task_type': TaskType.EXTRACT_DELIVERY,
        #     'example_file': EXTRACTION_FILES['deliv-example'],
        #     'test_file': EXTRACTION_FILES['deliv-test'],
        #     'output_file': EXTRACTION_FILES['deliv-output']
        # },
        # 'price': {
        #     'task_type': TaskType.EXTRACT_PRICE,
        #     'example_file': EXTRACTION_FILES['pri-example'],
        #     'test_file': EXTRACTION_FILES['pri-test'],
        #     'output_file': EXTRACTION_FILES['pri-output']
        # },
        # 'service': {
        #     'task_type': TaskType.EXTRACT_SERVICE,
        #     'example_file': EXTRACTION_FILES['ser-example'],
        #     'test_file': EXTRACTION_FILES['ser-test'],
        #     'output_file': EXTRACTION_FILES['ser-output']
        # }
    }

    print(f"=== Starting {aspect.upper()} Aspect Extraction ===")
    for aspect, config in extraction_config.items():
        print(f"\n--- Extracting {aspect.upper()} Aspects ---")
        generate(
            config['task_type'],
            example_file=config['example_file'],
            test_file=config['test_file'],
            output_file=config['output_file']
        )
        print(f"--- {aspect.upper()} Extraction Complete ---")
    print(f"\n=== {aspect.upper()} Aspect Extraction Complete ===")

# Uncomment to run extraction
run_extraction()

In [ ]:
# Run Delivery Aspect Extraction for delivery aspect only
def run_extraction():
    print("=== Starting Delivery Aspect Extraction ===")
    generate(
        TaskType.EXTRACT_DELIVERY,
        example_file=EXTRACTION_FILES['del-example'],
        test_file=EXTRACTION_FILES['del-test'],
        output_file=EXTRACTION_FILES['del-output']
    )
    print("\n=== Delivery Aspect Extraction Complete ===")

# Uncomment to run extraction
# run_extraction()

In [ ]:
# Run Price Aspect Extraction for price aspect only
def run_extraction():
    print("=== Starting Price Aspect Extraction ===")
    generate(
        TaskType.EXTRACT_PRICE,
        example_file=EXTRACTION_FILES['pri-example'],
        test_file=EXTRACTION_FILES['pri-test'],
        output_file=EXTRACTION_FILES['pri-output']
    )
    print("\n=== Price Aspect Extraction Complete ===")

# Uncomment to run extraction
run_extraction()

In [ ]:
# Run Service Aspect Extraction for service aspect only
def run_extraction():
    print("=== Starting Service Aspect Extraction ===")
    generate(
        TaskType.EXTRACT_SERVICE,
        example_file=EXTRACTION_FILES['ser-example'],
        test_file=EXTRACTION_FILES['ser-test'],
        output_file=EXTRACTION_FILES['ser-output']
    )
    print("\n=== Service Aspect Extraction Complete ===")

# Uncomment to run extraction
run_extraction()

In [ ]:
# Validate extractions
print("\n=== Starting Extraction Validation ===")
from aspect_extraction.extractor_validation import validate_extractions, save_f1_results
predicted = EXTRACTION_FILES['pri-output']
ground_truth = EXTRACTOR_FILES / "price-valid.json"
results = validate_extractions(predicted, ground_truth)
print(f"Average F1 Score: {results['average_f1']:.3f}")
print(f"Total reviews: {results['total_reviews']}")
print(f"Missing reviews: {results['missing_reviews']}")
print("\n=== Extraction Validation Complete ===")
# save_results(results, EXTRACTOR_FILES / "product_validation_results.txt")
# save_results(results, EXTRACTOR_FILES / "delivery_validation_results.txt")
save_f1_results(results, EXTRACTOR_FILES / "price_validation_results.txt")
# save_results(results, EXTRACTOR_FILES / "service_validation_results.txt")

In [ ]:
from aspect_extraction.extractor_validation import load_results_from_txt, extraction_error_analysis, save_error_analysis

error_analysis_results = extraction_error_analysis(load_results_from_txt(EXTRACTOR_FILES / "validation_results" / "price_validation_results.txt"))
save_error_analysis(error_analysis_results, EXTRACTOR_FILES / "error_analysis" / "price_error_analysis_results.txt")

## Visualization Generation
This section creates visualizations from validation outputs: confusion matrices (TP/FP/TN/FN labeled) and grouped bar charts comparing techniques across metrics (exact match accuracy, hamming loss, micro/macro F1). Run the visualization cells to display figures for each general aspect and for comparative technique-level charts.

Note: the visualizations use the aggregated `overall_performance` data and the per-aspect metrics defined below. Do not modify code cells—only run them to view the plots.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Plot confusion matrix for each general aspect
# gen_aspect = 'Product'
# aspect_metrics = {
#     'Color':        {'tp': 1, 'fp': 1, 'tn': 14, 'fn': 0},
#     'Condition':    {'tp': 2, 'fp': 2, 'tn': 11, 'fn': 1},
#     'Correctness':  {'tp': 0, 'fp': 2, 'tn': 13, 'fn': 1},
#     'Durability':   {'tp': 0, 'fp': 0, 'tn': 13, 'fn': 3},
#     'Effectivity':  {'tp': 0, 'fp': 0, 'tn': 14, 'fn': 2},
#     'Functionality':{'tp': 3, 'fp': 2, 'tn': 6, 'fn': 5},
#     'Material':     {'tp': 0, 'fp': 0, 'tn': 16, 'fn': 0},
#     'Sensory':      {'tp': 1, 'fp': 0, 'tn': 15, 'fn': 0},
#     'Size':         {'tp': 3, 'fp': 0, 'tn': 12, 'fn': 1},
#     'General':      {'tp': 0, 'fp': 0, 'tn': 13, 'fn': 3}
# }

# gen_aspect = 'Delivery'
# aspect_metrics = {
#     'Condition':    {'tp': 0, 'fp': 1, 'tn': 9, 'fn': 0},
#     'Correctness':  {'tp': 5, 'fp': 1, 'tn': 1, 'fn': 3},
#     'Timeliness':   {'tp': 1, 'fp': 4, 'tn': 5, 'fn': 0},
#     'General':      {'tp': 0, 'fp': 0, 'tn': 9, 'fn': 1}
# }

gen_aspect = 'Price'
aspect_metrics = {
    'Affordability':    {'tp': 2, 'fp': 0, 'tn': 6, 'fn': 0},
    'Value_for_Money':  {'tp': 6, 'fp': 0, 'tn': 2, 'fn': 0},
    'General':          {'tp': 0, 'fp': 0, 'tn': 8, 'fn': 0}
}

# gen_aspect = 'Service'
# aspect_metrics = {
#     'Handling':         {'tp': 1, 'fp': 0, 'tn': 5, 'fn': 2},
#     'Responsiveness':   {'tp': 2, 'fp': 0, 'tn': 4, 'fn': 2},
#     'Trustworthiness':  {'tp': 2, 'fp': 4, 'tn': 2, 'fn': 0},
#     'General':          {'tp': 0, 'fp': 1, 'tn': 7, 'fn': 0}
# }

# Create subplots for all confusion matrices
#fig, axes = plt.subplots(2, 5, figsize=(20, 7))   #Product
# fig, axes = plt.subplots(2, 2, figsize=(10, 8))  #Service & Delivery
fig, axes = plt.subplots(1, 3, figsize=(17, 5))  #Price
axes = axes.flatten()

for idx, (aspect, metrics) in enumerate(aspect_metrics.items()):
    tp, fp, tn, fn = metrics['tp'], metrics['fp'], metrics['tn'], metrics['fn']
    
    # Create the confusion matrix
    labels = np.array([[f'TN\n{tn}', f'FP\n{fp}'],
                      [f'FN\n{fn}', f'TP\n{tp}']])
    cm = np.array([[tn, fp], [fn, tp]])
    
    # Create heatmap on the specific subplot
    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues',
                xticklabels=['Negative', 'Positive'],
                yticklabels=['Negative', 'Positive'],
                ax=axes[idx], cbar=True)
    
    axes[idx].set_title(f'Rule-Based: {gen_aspect}-{aspect}')
    axes[idx].set_ylabel('True Label')
    axes[idx].set_xlabel('Predicted Label')
    
    # Center the annotations
    for t in axes[idx].texts:
        t.set_horizontalalignment('center')

plt.tight_layout()
plt.savefig(f'{gen_aspect}_specific_cm_rb.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_all_metrics_bar_charts(metrics_dict, aspects, techniques):
    """
    Plot bar charts for all metrics across all aspects and techniques.
    """
    metric_names = ['Exact Match Accuracy', 'Hamming Loss', 'Micro-average F1', 'Macro-average F1']
    colors = ['lightblue', 'salmon', 'lightgreen', 'plum']

    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    axes = axes.flatten()

    for idx, aspect in enumerate(aspects):
        ax = axes[idx]
        metrics = metrics_dict[aspect]
        x = np.arange(len(techniques))
        width = 0.2

        bars = []
        for i, metric in enumerate(metric_names):
            offset = width * (i - 1.5)
            bar = ax.bar(x + offset, metrics[metric], width, label=metric, color=colors[i])
            bars.append(bar)

        ax.set_ylabel('Score', fontsize=10)
        ax.set_title(f'Aspect Identification of {aspect} Specific Aspects Performance Metrics Comparison')
        ax.set_xticks(x)
        ax.set_xticklabels(techniques, rotation=15, ha='right', fontsize=9)
        ax.legend(fontsize=8, loc='upper right')
        ax.set_ylim(0, 1.1)
        ax.grid(axis='y', alpha=0.3)

        for bar in bars:
            for rect in bar:
                height = rect.get_height()
                ax.annotate(f'{height:.2f}',
                            xy=(rect.get_x() + rect.get_width()/2, height),
                            xytext=(0, 2),
                            textcoords="offset points",
                            ha='center', va='bottom', rotation=0)
                
    fig.suptitle('Specific Aspects Performance Metrics Comparison', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    plt.show()

# Example usage:
techniques = ['LLM (Gemini)', 'Rule-based', 'Fine-Tuned (Gemini DS)', 'Fine-Tuned (Rule-based DS)']
aspects = ['Product', 'Delivery', 'Price', 'Service']
# GENERAL ASPECTS
# metrics = {
#     'Exact Match Accuracy': [0.7826, 0.5217, 0.0870, 0.1304],
#     'Hamming Loss': [0.0761, 0.1630, 0.5543, 0.4783],
#     'Micro-average F1': [0.9136, 0.8200, 0.6047, 0.6452],
#     'Macro-average F1': [0.9243, 0.8400, 0.5888, 0.6282]
# }
metrics_dict = {
    'Product': {
        'Exact Match Accuracy': [0.2500, 0.1250, 0.0000, 0.0000],
        'Hamming Loss': [0.1000, 0.1437, 0.4375, 0.5250],
        'Micro-average F1': [0.6923, 0.4700, 0.3396, 0.2075],
        'Macro-average F1': [0.5266, 0.3600, 0.1722, 0.1173]
    },
    'Delivery': {
        'Exact Match Accuracy': [0.9000, 0.2000, 0.1000, 0.1000],
        'Hamming Loss': [0.0250, 0.2500, 0.4250, 0.4000],
        'Micro-average F1': [0.9474, 0.5500, 0.1053, 0.1111],
        'Macro-average F1': [0.5000, 0.2600, 0.0556, 0.0556]
    },
    'Price': {
        'Exact Match Accuracy': [0.875, 1.000, 0.0000, 0.0000],
        'Hamming Loss': [0.0833, 0.0000, 0.6250, 0.5000],
        'Micro-average F1': [0.8750, 1.0000, 0.0000, 0.1111],
        'Macro-average F1': [0.5697, 0.6667, 0.0000, 0.1250]
    },
    'Service': {
        'Exact Match Accuracy': [0.5000, 0.2500, 0.0000, 0.0000],
        'Hamming Loss': [0.1562, 0.2812, 0.6562, 0.3333],
        'Micro-average F1': [0.7619, 0.5625, 0.1600, 0.0000],
        'Macro-average F1': [0.5764, 0.4688, 0.1000, 0.0000]
    }
}

plot_all_metrics_bar_charts(metrics_dict, aspects, techniques)